<a href="https://colab.research.google.com/github/postnicov/ResazurinResorufin/blob/main/Resazurin_Resorufin_Color_Mixing_Reflection_v6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Resazurin / Resorufin pH-dependent color mixing (Reflection, v6)

This Colab notebook retrieves pH-dependent resazurin and resorufin absorbance spectra, mixes them at selected resorufin fractions, converts the mixtures to diffuse reflectance using wavelength-dependent Kubelka-Munk scattering, and exports CIE L*a*b*, sRGB, and Munsell Illuminant C results to Excel.

**v6 update:** The Excel file now contains the original, non-rounded Munsell(C) notation; its 0.5-rounded notation; a numeric hue coordinate `H`; and unrounded Munsell chroma `C`. The hue coordinate uses the cyclic sequence B → PB → P → RP → R → YR → Y → GY → G → BG → B, with integer sector indices 1–10 and the Munsell hue number as the fractional component.

In [1]:
#@title Install and import required packages
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    'colour-science': 'colour',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'openpyxl': 'openpyxl',
    'XlsxWriter': 'xlsxwriter',
}

missing_packages = [
    package for package, module in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    print('Installing:', ', '.join(missing_packages))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing_packages])
else:
    print('All required packages are already installed.')

import re
import warnings

import colour
import numpy as np
import pandas as pd
from colour.notation import xyY_to_munsell_colour
from colour.adaptation import chromatic_adaptation_VonKries
from colour.utilities import ColourUsageWarning

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

warnings.filterwarnings('ignore', category=ColourUsageWarning)
print('Libraries imported successfully.')

Installing: colour-science, XlsxWriter
Libraries imported successfully.


In [2]:
#@title Set data source URLs and load the spectra tables
RESAZURIN_URL = 'https://raw.githubusercontent.com/postnicov/ResazurinResorufin/refs/heads/main/Data/pHRzSpectra.csv'  #@param {type:"string"}
RESORUFIN_URL = 'https://raw.githubusercontent.com/postnicov/ResazurinResorufin/refs/heads/main/Data/pHRfSpectra.csv'  #@param {type:"string"}

def load_spectra(url):
    """Load a spectra CSV whose first row gives pH and first column gives wavelength."""
    raw = pd.read_csv(url, header=0, index_col=0)
    raw.columns = [float(column) for column in raw.columns]
    raw.index = raw.index.astype(float)
    return raw.sort_index()

resazurin_spectra = load_spectra(RESAZURIN_URL)
resorufin_spectra = load_spectra(RESORUFIN_URL)
common_pH = sorted(set(resazurin_spectra.columns).intersection(resorufin_spectra.columns))
if not common_pH:
    raise ValueError('No common pH values were found between the two spectra tables.')

print(f'Resazurin spectrum: {resazurin_spectra.shape[0]} wavelengths x {resazurin_spectra.shape[1]} pH values')
print(f'Resorufin spectrum: {resorufin_spectra.shape[0]} wavelengths x {resorufin_spectra.shape[1]} pH values')
print(f'Common pH values ({len(common_pH)}):', common_pH)

Resazurin spectrum: 341 wavelengths x 17 pH values
Resorufin spectrum: 341 wavelengths x 17 pH values
Common pH values (17): [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0]


In [3]:
#@title Set the resorufin fraction range (x)
X_MIN = 0.05  #@param {type:"number"}
X_STEP = 0.1  #@param {type:"number"}
X_MAX = 0.95  #@param {type:"number"}

if X_STEP <= 0:
    raise ValueError('X_STEP must be positive.')
if X_MAX < X_MIN:
    raise ValueError('X_MAX must be greater than or equal to X_MIN.')

n_steps = int(round((X_MAX - X_MIN) / X_STEP))
x_values = [round(X_MIN + i * X_STEP, 10) for i in range(n_steps + 1)]
if x_values[-1] < X_MAX and not np.isclose(x_values[-1], X_MAX):
    x_values.append(round(X_MAX, 10))
x_values = sorted(set(x_values))
if any(x < 0 or x > 1 for x in x_values):
    raise ValueError('All x values must lie between 0 and 1.')

print(f'Resorufin fraction values x ({len(x_values)}):', x_values)

Resorufin fraction values x (10): [0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85, 0.95]


In [4]:
#@title Set wavelength-dependent Kubelka-Munk scattering parameters
S0 = 16.185761  #@param {type:"number"}
s = 0.001005  #@param {type:"number"}

if S0 <= 0:
    raise ValueError('S0 must be positive.')
if s < 0:
    raise ValueError('s must be non-negative.')

print('Scattering function: S(lambda) = S0 * exp[-s * (lambda - 360)]')
print(f'S0 = {S0}; s = {s}')

Scattering function: S(lambda) = S0 * exp[-s * (lambda - 360)]
S0 = 16.185761; s = 0.001005


In [5]:
#@title Set Munsell Illuminant C chromatic-adaptation settings
OBSERVER = 'CIE 1931 2 Degree Standard Observer'
CAT_TRANSFORM = 'Bradford'  #@param ['Bradford', 'CAT02', 'CAT16', 'Von Kries']

xy_C = colour.CCS_ILLUMINANTS[OBSERVER]['C']
xy_D65 = colour.CCS_ILLUMINANTS[OBSERVER]['D65']
XYZ_w_C = colour.xy_to_XYZ(xy_C)
XYZ_w_D65 = colour.xy_to_XYZ(xy_D65)

print('Munsell renotation illuminant: C')
print('Chromatic adaptation: Von Kries with', CAT_TRANSFORM, 'transform')
print('C white point xy:', xy_C)
print('D65 white point xy:', xy_D65)

Munsell renotation illuminant: C
Chromatic adaptation: Von Kries with Bradford transform
C white point xy: [ 0.31006  0.31616]
D65 white point xy: [ 0.3127  0.329 ]


In [6]:
#@title Define spectral mixing, Kubelka-Munk, colorimetry, and Munsell Illuminant C functions
ILLUMINANT = colour.SDS_ILLUMINANTS['D65']
CMFS = colour.MSDS_CMFS[OBSERVER]
D65_XY = xy_D65

def mix_absorbance(abs_rz, abs_rf, x):
    """Linearly mix absorbance: (1 - x) * resazurin + x * resorufin."""
    return (1.0 - x) * abs_rz + x * abs_rf

def scattering_coefficient(wavelengths, S0, s):
    """Return S(lambda) = S0 * exp[-s * (lambda - 360)]."""
    wavelengths = np.asarray(wavelengths, dtype=float)
    return S0 * np.exp(-s * (wavelengths - 360.0))

def absorbance_to_reflectance(absorbance, wavelengths, S0, s):
    """Use K(lambda)/S(lambda) in the optically thick Kubelka-Munk reflectance relation."""
    scattering = scattering_coefficient(wavelengths, S0, s)
    k_over_s = np.clip(np.asarray(absorbance, dtype=float), 0.0, None) / scattering
    reflectance = 1.0 + k_over_s - np.sqrt(k_over_s ** 2 + 2.0 * k_over_s)
    return np.clip(reflectance, 0.0, 1.0)

def spectrum_to_lab_rgb(wavelengths, reflectance):
    """Convert reflectance to CIE Lab, 8-bit sRGB, and XYZ under D65."""
    sd = colour.SpectralDistribution(dict(zip(wavelengths, reflectance)))
    sd = sd.align(colour.SpectralShape(int(wavelengths.min()), int(wavelengths.max()), 1))
    xyz_d65 = colour.sd_to_XYZ(sd, cmfs=CMFS, illuminant=ILLUMINANT) / 100.0
    lab = colour.XYZ_to_Lab(xyz_d65, D65_XY)
    rgb = colour.XYZ_to_sRGB(xyz_d65)
    rgb_8bit = np.clip(np.round(rgb * 255.0), 0, 255).astype(int)
    return lab, rgb_8bit, xyz_d65

HUE_SECTORS = ['R', 'YR', 'Y', 'GY', 'G', 'BG', 'B', 'PB', 'P', 'RP']
HUE_COORDINATE_SECTORS = {'B': 1, 'PB': 2, 'P': 3, 'RP': 4, 'R': 5, 'YR': 6, 'Y': 7, 'GY': 8, 'G': 9, 'BG': 10}

def format_munsell_number(number):
    """Format a Munsell number without a trailing decimal zero where integral."""
    number = float(number)
    if np.isclose(number, round(number)):
        return str(int(round(number)))
    return f'{number:.2f}'.rstrip('0').rstrip('.')

def normalise_zero_hue_to_previous_sector(munsell_notation, tolerance=1e-8):
    """Convert a zero hue in one sector to 10 of the preceding sector, e.g. 0PB to 10B."""
    text = str(munsell_notation).strip().upper()
    match = re.fullmatch(
        r'([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*([A-Z]+)\s+([^\s/]+)\s*/\s*([^\s]+)',
        text,
    )
    if match is None:
        raise ValueError(f'Cannot parse Munsell notation: {munsell_notation!r}')
    hue_number_text, sector, value_text, chroma_text = match.groups()
    hue_number = float(hue_number_text)
    if np.isclose(hue_number, 0.0, atol=tolerance) and sector in HUE_SECTORS:
        prior_sector = HUE_SECTORS[(HUE_SECTORS.index(sector) - 1) % len(HUE_SECTORS)]
        return f'10{prior_sector} {value_text}/{chroma_text}'
    return f'{format_munsell_number(hue_number)}{sector} {value_text}/{chroma_text}'

def xyz_d65_to_munsell_c(xyz_d65):
    """Adapt XYZ(D65) to XYZ(C), return unrounded Munsell(C) notation and a correction note."""
    xyz_d65 = np.asarray(xyz_d65, dtype=float)
    xyz_c = chromatic_adaptation_VonKries(
        xyz_d65, XYZ_w_D65, XYZ_w_C, transform=CAT_TRANSFORM
    )
    try:
        raw_munsell = xyY_to_munsell_colour(colour.XYZ_to_xyY(xyz_c))
        return normalise_zero_hue_to_previous_sector(raw_munsell), ''
    except Exception:
        pass

    lab_d65 = colour.XYZ_to_Lab(xyz_d65, D65_XY)
    clipped_lab = np.array([np.clip(lab_d65[0], 10.0, 90.0), lab_d65[1], lab_d65[2]])
    for chroma_scale in np.linspace(1.0, 0.0, 101):
        candidate_lab_d65 = np.array([
            clipped_lab[0],
            clipped_lab[1] * chroma_scale,
            clipped_lab[2] * chroma_scale,
        ])
        candidate_xyz_d65 = colour.Lab_to_XYZ(candidate_lab_d65, D65_XY)
        candidate_xyz_c = chromatic_adaptation_VonKries(
            candidate_xyz_d65, XYZ_w_D65, XYZ_w_C, transform=CAT_TRANSFORM
        )
        try:
            raw_munsell = xyY_to_munsell_colour(colour.XYZ_to_xyY(candidate_xyz_c))
            return normalise_zero_hue_to_previous_sector(raw_munsell), 'nearest valid'
        except Exception:
            continue
    return 'Unavailable', 'nearest valid'

def round_to_half(value):
    """Round a positive decimal to nearest 0.5 using half-up rounding."""
    return np.floor(float(value) * 2.0 + 0.5) / 2.0

def format_half_step(value):
    """Format a half-step without .0 for whole numbers."""
    rounded_value = round_to_half(value)
    if np.isclose(rounded_value, round(rounded_value)):
        return str(int(round(rounded_value)))
    return f'{rounded_value:.1f}'

def parse_munsell_components(munsell_code):
    """Parse a normalized Munsell code into hue number, sector, value, and unrounded chroma."""
    match = re.fullmatch(
        r'([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*([A-Z]+)\s+([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*/\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+))',
        str(munsell_code).strip().upper(),
    )
    if match is None:
        return np.nan, None, np.nan, np.nan
    hue_number, hue_sector, value, chroma = match.groups()
    return float(hue_number), hue_sector, float(value), float(chroma)

def munsell_hue_coordinate(munsell_code):
    """Map Munsell hue to H = sector index + hue number / 10.

    Sector order: B=1, PB=2, P=3, RP=4, R=5, YR=6, Y=7, GY=8, G=9, BG=10.
    Thus 10B maps to 2.0, which is identical to 0PB before boundary normalization.
    """
    hue_number, hue_sector, _, _ = parse_munsell_components(munsell_code)
    if hue_sector not in HUE_COORDINATE_SECTORS or not np.isfinite(hue_number):
        return np.nan
    return HUE_COORDINATE_SECTORS[hue_sector] + hue_number / 10.0

def munsell_chroma(munsell_code):
    """Return the unrounded chroma component from a Munsell notation."""
    _, _, _, chroma = parse_munsell_components(munsell_code)
    return chroma

def munsell_to_half_step(munsell_code):
    """Round a Munsell(C) hue, value, and chroma to the nearest 0.5."""
    neutral_match = re.fullmatch(r'N\s+(\d+(?:\.\d+)?)/', munsell_code)
    if neutral_match:
        return f'N {format_half_step(neutral_match.group(1))}/'
    hue, hue_sector, value, chroma = parse_munsell_components(munsell_code)
    if hue_sector is None:
        return munsell_code
    return f'{format_half_step(hue)}{hue_sector} {format_half_step(value)}/{format_half_step(chroma)}'

print('Core functions defined.')

Core functions defined.


In [7]:
#@title Compute mixed spectra, colors, and Munsell Illuminant C values for all pH and x combinations
wavelengths_rz = resazurin_spectra.index.to_numpy(dtype=float)
wavelengths_rf = resorufin_spectra.index.to_numpy(dtype=float)
common_wavelengths = np.intersect1d(wavelengths_rz, wavelengths_rf)
if len(common_wavelengths) == 0:
    raise ValueError('The two spectra tables have no wavelengths in common.')

resazurin_spectra = resazurin_spectra.loc[common_wavelengths]
resorufin_spectra = resorufin_spectra.loc[common_wavelengths]
wavelengths = common_wavelengths.astype(float)
S_lambda = scattering_coefficient(wavelengths, S0, s)

results = []
for pH in common_pH:
    abs_rz = resazurin_spectra[pH].to_numpy(dtype=float)
    abs_rf = resorufin_spectra[pH].to_numpy(dtype=float)
    for x in x_values:
        mixed_abs = mix_absorbance(abs_rz, abs_rf, x)
        reflectance = absorbance_to_reflectance(mixed_abs, wavelengths, S0, s)
        lab, rgb, xyz_d65 = spectrum_to_lab_rgb(wavelengths, reflectance)
        munsell_raw, munsell_note = xyz_d65_to_munsell_c(xyz_d65)
        results.append({
            'pH': pH,
            'x': x,
            'L*': round(float(lab[0]), 3),
            'a*': round(float(lab[1]), 3),
            'b*': round(float(lab[2]), 3),
            'R': int(rgb[0]),
            'G': int(rgb[1]),
            'B': int(rgb[2]),
            'Munsell (C), unrounded': munsell_raw,
            'Munsell (C), 0.5': munsell_to_half_step(munsell_raw),
            'H': munsell_hue_coordinate(munsell_raw),
            'C': munsell_chroma(munsell_raw),
            'Munsell note': munsell_note,
        })

results_df = pd.DataFrame(results)
print(f'Computed {len(results_df)} pH × x combinations using {len(wavelengths)} common wavelengths.')
print(f'S(lambda) range across the spectrum: {S_lambda.min():.6f} to {S_lambda.max():.6f}')

reference = results_df[np.isclose(results_df['pH'], 6.5) & np.isclose(results_df['x'], 0.0)]
if not reference.empty:
    print('Check — pH = 6.5, x = 0, Munsell (C), 0.5:', reference.iloc[0]['Munsell (C), 0.5'])
display(results_df.head(10))

Computed 170 pH × x combinations using 341 common wavelengths.
S(lambda) range across the spectrum: 11.500976 to 16.185761


,pH,x,L*,a*,b*,R,G,B,"Munsell (C), unrounded","Munsell (C), 0.5",H,C,Munsell note
0,1.0,0.05,94.802,8.562,2.687,255,234,235,1.2R 9.5/3.7,1R 9.5/3.5,5.12,3.7,
1,1.0,0.15,94.967,8.133,2.913,255,235,235,2R 9.5/3.7,2R 9.5/3.5,5.20,3.7,
2,1.0,0.25,95.141,7.692,3.151,255,236,235,3R 9.5/3.6,3R 9.5/3.5,5.30,3.6,
3,1.0,0.35,95.323,7.234,3.401,255,237,235,4R 9.5/3.5,4R 9.5/3.5,5.40,3.5,
4,1.0,0.45,95.516,6.757,3.666,255,238,235,5.2R 9.5/3.4,5R 9.5/3.5,5.52,3.4,
5,1.0,0.55,95.720,6.257,3.949,255,238,235,6.5R 9.6/3.3,6.5R 9.5/3.5,5.65,3.3,
6,1.0,0.65,95.938,5.731,4.252,255,239,235,7.9R 9.6/3.2,8R 9.5/3,5.79,3.2,
7,1.0,0.75,96.173,5.171,4.580,255,240,235,9.5R 9.6/3.2,9.5R 9.5/3,5.95,3.2,
8,1.0,0.85,96.428,4.571,4.941,255,241,235,1.2YR 9.6/3.1,1YR 9.5/3,6.12,3.1,
9,1.0,0.95,96.713,3.920,5.347,255,243,235,3YR 9.7/3.1,3YR 9.5/3,6.30,3.1,


In [8]:
#@title Build and download the Excel file with colored swatches and Munsell Illuminant C values
from datetime import datetime

OUTPUT_FILENAME = f"Resazurin_Resorufin_colors_Reflection_v6_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"

export_df = results_df[['pH', 'x', 'L*', 'a*', 'b*']].copy()
export_df['Color'] = ''
export_df['R'] = results_df['R']
export_df['G'] = results_df['G']
export_df['B'] = results_df['B']
export_df['Munsell (C), unrounded'] = results_df['Munsell (C), unrounded']
export_df['Munsell (C), 0.5'] = results_df['Munsell (C), 0.5']
export_df['H'] = results_df['H']
export_df['C'] = results_df['C']
export_df['Munsell note'] = results_df['Munsell note']

output_path = '/content/' + OUTPUT_FILENAME if IN_COLAB else OUTPUT_FILENAME

with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
    export_df.to_excel(writer, sheet_name='Colors', index=False)
    workbook = writer.book
    worksheet = writer.sheets['Colors']
    header_format = workbook.add_format({'bold': True, 'align': 'center', 'valign': 'vcenter', 'border': 1})
    numeric_format = workbook.add_format({'num_format': '0.0000'})
    for column_index, column_name in enumerate(export_df.columns):
        worksheet.write(0, column_index, column_name, header_format)

    color_column = export_df.columns.get_loc('Color')
    h_column = export_df.columns.get_loc('H')
    c_column = export_df.columns.get_loc('C')
    for row_index, (_, row) in enumerate(results_df.iterrows(), start=1):
        hex_color = '#{:02X}{:02X}{:02X}'.format(int(row['R']), int(row['G']), int(row['B']))
        swatch_format = workbook.add_format({'bg_color': hex_color, 'border': 1})
        worksheet.write_blank(row_index, color_column, None, swatch_format)
        worksheet.write_number(row_index, h_column, float(row['H']) if np.isfinite(row['H']) else np.nan, numeric_format)
        worksheet.write_number(row_index, c_column, float(row['C']) if np.isfinite(row['C']) else np.nan, numeric_format)

    worksheet.set_column('A:A', 8)
    worksheet.set_column('B:B', 8)
    worksheet.set_column('C:E', 10)
    worksheet.set_column('F:F', 12)
    worksheet.set_column('G:I', 8)
    worksheet.set_column('J:K', 23)
    worksheet.set_column('L:M', 11)
    worksheet.set_column('N:N', 16)

print(f'Excel file created: {output_path}')
if IN_COLAB:
    files.download(output_path)
display(export_df.head(10))

Excel file created: /content/Resazurin_Resorufin_colors_Reflection_v6_20260821_075335.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,pH,x,L*,a*,b*,Color,R,G,B,"Munsell (C), unrounded","Munsell (C), 0.5",H,C,Munsell note
0,1.0,0.05,94.802,8.562,2.687,,255,234,235,1.2R 9.5/3.7,1R 9.5/3.5,5.12,3.7,
1,1.0,0.15,94.967,8.133,2.913,,255,235,235,2R 9.5/3.7,2R 9.5/3.5,5.20,3.7,
2,1.0,0.25,95.141,7.692,3.151,,255,236,235,3R 9.5/3.6,3R 9.5/3.5,5.30,3.6,
3,1.0,0.35,95.323,7.234,3.401,,255,237,235,4R 9.5/3.5,4R 9.5/3.5,5.40,3.5,
4,1.0,0.45,95.516,6.757,3.666,,255,238,235,5.2R 9.5/3.4,5R 9.5/3.5,5.52,3.4,
5,1.0,0.55,95.720,6.257,3.949,,255,238,235,6.5R 9.6/3.3,6.5R 9.5/3.5,5.65,3.3,
6,1.0,0.65,95.938,5.731,4.252,,255,239,235,7.9R 9.6/3.2,8R 9.5/3,5.79,3.2,
7,1.0,0.75,96.173,5.171,4.580,,255,240,235,9.5R 9.6/3.2,9.5R 9.5/3,5.95,3.2,
8,1.0,0.85,96.428,4.571,4.941,,255,241,235,1.2YR 9.6/3.1,1YR 9.5/3,6.12,3.1,
9,1.0,0.95,96.713,3.920,5.347,,255,243,235,3YR 9.7/3.1,3YR 9.5/3,6.30,3.1,
